In [ ]:
• 🔍 SECTION-BY-SECTION DATA TRACE

  I'll trace exactly how one batch flows through the system, showing data at every transformation.

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 1: Document Extraction from Parquet

  📐 Code

  def _document_batches(split, resume_state_dict, tokenizer_batch_size):
      parquet_paths = list_parquet_files()
      parquet_paths = parquet_paths[:-1] if split == "train" else parquet_paths[-1:]

      for pq_idx in range(len(parquet_paths)):
          pf = pq.ParquetFile(filepath)
          rg_idx = ddp_rank  # Each GPU starts at different row_group
          while rg_idx < pf.num_row_groups:
              rg = pf.read_row_group(rg_idx)
              batch = rg.column('text').to_pylist()
              for i in range(0, len(batch), tokenizer_batch_size):
                  yield batch[i:i+tokenizer_batch_size], (pq_idx, rg_idx, epoch)
              rg_idx += ddp_world_size  # Skip to next for this rank

  🧮 Data Trace

  ┌────────────────────────────────────────────────────────────────────┐
  │  PARQUET FILE: shard_00000.parquet                                 │
  │  ┌──────────────────────────────────────────────────────────────┐  │
  │  │ Row Group 0 (GPU 0)   Row Group 1 (GPU 1)   Row Group 2 (GPU 2)│ │
  │  │ ┌───────────────┐    ┌───────────────┐    ┌───────────────┐   │ │
  │  │ │ "The cat..."  │    │ "In 1492..."  │    │ "Python is.." │   │ │
  │  │ │ "Quantum..."  │    │ "The stock.." │    │ "Neural net.."│   │ │
  │  │ │   ...1000 docs│    │   ...1000 docs│    │   ...1000 docs│   │ │
  │  │ └──────┬────────┘    └──────┬────────┘    └──────┬────────┘   │ │
  │  └────────┼────────────────────┼────────────────────┼────────────┘ │
  └───────────┼────────────────────┼────────────────────┼──────────────┘
              │                    │                    │
              ▼                    ▼                    ▼
        GPU 0 reads           GPU 1 reads           GPU 2 reads
        row_group 0           row_group 1           row_group 2
        then 3, 6...          then 4, 7...          then 5, 8...

  YIELD (per batch of 128 docs):
    doc_batch = [
      "The cat sat on the mat and looked at the moon.",
      "In 1492, Christopher Columbus sailed the ocean blue.",
      "def hello_world():\n    print('Hello')",
      ... 125 more docs
    ]
    state = (pq_idx=0, rg_idx=0, epoch=1)

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 2: Tokenization with BOS Prepending

  📐 Code

  def refill_buffer():
      doc_batch, (pq_idx, rg_idx, epoch) = next(batches)
      # Tokenize with BOS prepended
      token_lists = tokenizer.encode(doc_batch, prepend=bos_token, num_threads=4)
      for tokens in token_lists:
          doc_buffer.append(tokens)

  🧮 Data Trace

  INPUT: doc_batch (list of strings)
    [0] "The cat sat on the mat"
    [1] "In 1492, Columbus sailed"
    [2] "def hello():"
    [3] "Quantum mechanics is..."

  TOKENIZATION (with BOS=1):
    doc_buffer after refill:

    ┌────────┬──────────────────────────────────────┬────────┐
    │ Doc ID │ Tokens (first 10 shown)              │ Length │
    ├────────┼──────────────────────────────────────┼────────┤
    │   0    │ [1, 567, 892, 123, 45, 678, 234]     │   7    │
    │   1    │ [1, 89, 4567, 2345, 1234, 567]       │   6    │
    │   2    │ [1, 345, 678, 90]                    │   3    │
    │   3    │ [1, 1234, 5678, 9012, 3456, 789, ...│  45    │
    │  ...   │ ...                                  │  ...   │
    └────────┴──────────────────────────────────────┴────────┘
                ↑
             BOS token (ID=1) prepended to EVERY document

    Buffer size: 1000 tokenized documents ready for packing

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 3: Best-Fit Packing Algorithm (THE CORE)

  📐 Code

  for row_idx in range(B):        # For each row in batch
      pos = 0                      # Current position in row
      while pos < row_capacity:    # Fill until full (T+1)
          remaining = row_capacity - pos

          # BEST-FIT: Find largest doc that fits entirely
          best_idx = -1
          best_len = 0
          for i, doc in enumerate(doc_buffer):
              if len(doc) <= remaining and len(doc) > best_len:
                  best_idx = i
                  best_len = len(doc)

          if best_idx >= 0:
              doc = doc_buffer.pop(best_idx)  # Remove from buffer
              row_buffer[row_idx, pos:pos+len(doc)] = torch.tensor(doc)
              pos += len(doc)
          else:
              # CROP: No doc fits, crop shortest to fill exactly
              shortest_idx = min(range(len(doc_buffer)), key=lambda i: len(doc_buffer[i]))
              doc = doc_buffer.pop(shortest_idx)
              row_buffer[row_idx, pos:pos+remaining] = torch.tensor(doc[:remaining])
              pos += remaining  # Row is now full

  🧮 Data Trace: Building Row 0

  CONFIG: B=2, T=8 → row_capacity=9 tokens

  INITIAL STATE:
    doc_buffer = [
      0: [1, 567, 892, 123, 45, 678, 234]     len=7   (doc0)
      1: [1, 89, 4567, 2345, 1234, 567]       len=6   (doc1)
      2: [1, 345, 678, 90]                    len=3   (doc2)
      3: [1, 1234, 5678, 9012, ...]           len=45  (doc3 - too big!)
      4: [1, 55, 66, 77]                      len=4   (doc4)
      ...
    ]

    row_buffer[0] = [_, _, _, _, _, _, _, _, _]  (empty, 9 slots)
    pos = 0, remaining = 9

  ═══════════════════════════════════════════════════════════════════
  ITERATION 1: Find largest doc with len ≤ 9

    Scanning buffer:
      doc0: len=7 ≤ 9 ✓, best so far
      doc1: len=6 ≤ 9 ✓, but 6 < 7, skip
      doc2: len=3 ≤ 9 ✓, but 3 < 7, skip
      doc3: len=45 > 9 ✗, doesn't fit
      doc4: len=4 ≤ 9 ✓, but 4 < 7, skip

    BEST: doc0 (len=7)

    ACTION:
      Pop doc0 from buffer
      row_buffer[0, 0:7] = [1, 567, 892, 123, 45, 678, 234]

    RESULT:
      row_buffer[0] = [1, 567, 892, 123, 45, 678, 234, _, _]
                                                    ↑
                                                 pos=7
      remaining = 9 - 7 = 2

  ═══════════════════════════════════════════════════════════════════
  ITERATION 2: Find largest doc with len ≤ 2

    Scanning buffer:
      doc1: len=6 > 2 ✗
      doc2: len=3 > 2 ✗
      doc4: len=4 > 2 ✗
      ...nothing fits!

    BEST: None found → MUST CROP

    ACTION (crop shortest):
      Find shortest: doc2 (len=3)
      Take first 2 tokens: [1, 345]
      Pop doc2 from buffer
      row_buffer[0, 7:9] = [1, 345]

    RESULT:
      row_buffer[0] = [1, 567, 892, 123, 45, 678, 234, 1, 345]
                                                    ↑
                                                 pos=9
      remaining = 0 → ROW FULL!

  ═══════════════════════════════════════════════════════════════════
  FINAL ROW 0:
    [1, 567, 892, 123, 45, 678, 234, 1, 345]
     └──────doc0─────────┘└doc2┘
     (7 tokens)           (2 tokens, cropped)

    DISCARDED from doc2: [678, 90] → will never be trained on!

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 4: Building All Rows in Batch

  🧮 Data Trace: Complete Batch

  After filling all B=2 rows:

  row_buffer (CPU, shape [2, 9]):
  ┌─────────────────────────────────────────────────────────────┐
  │ ROW 0: [1, 567, 892, 123, 45, 678, 234, 1, 345]            │
  │         │  └────────────────────────┘  │  └─┘              │
  │        BOS         doc0 (7 tokens)    BOS doc2 cropped      │
  │                                                             │
  │ ROW 1: [1, 89, 4567, 2345, 1, 345, 678, 90, 1]             │
  │         │  └────────────┘  └──────────┘     │              │
  │        BOS     doc1 (6)       doc4 (4)     BOS doc5 crop   │
  │                                                             │
  │  BOS positions: Every document boundary has BOS!            │
  └─────────────────────────────────────────────────────────────┘

  TOTAL TOKENS: 18 tokens across 2 rows
  CROPPED: ~4 tokens (from docs 2 and 5)
  CROP RATE: ~22% (nanochat reports ~35% at T=2048)

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 5: Input/Target Split (Autoregressive Setup)

  📐 Code

  # Slice into inputs and targets (shifted by 1)
  cpu_inputs.copy_(row_buffer[:, :-1])   # All except last
  cpu_targets.copy_(row_buffer[:, 1:])   # All except first

  🧮 Data Trace

  INPUT: row_buffer (shape [2, 9])
    ROW 0: [1, 567, 892, 123, 45, 678, 234, 1, 345]
    ROW 1: [1, 89, 4567, 2345, 1, 345, 678, 90, 1]

  SPLITTING:
  ┌─────────────────────────────────────────────────────────────┐
  │ cpu_inputs = row_buffer[:, :-1]  → shape [2, 8]             │
  │                                                             │
  │ ROW 0: [1, 567, 892, 123, 45, 678, 234, 1]                 │
  │ ROW 1: [1, 89, 4567, 2345, 1, 345, 678, 90]                │
  │              ↑                                              │
  │         Last token of row 0 is dropped (no target)         │
  └─────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────┐
  │ cpu_targets = row_buffer[:, 1:]  → shape [2, 8]             │
  │                                                             │
  │ ROW 0: [567, 892, 123, 45, 678, 234, 1, 345]               │
  │ ROW 1: [89, 4567, 2345, 1, 345, 678, 90, 1]                │
  │         ↑                                                   │
  │    First token (BOS) dropped - never predicted!             │
  └─────────────────────────────────────────────────────────────┘

  AUTOREGRESSIVE PAIRS (what the model learns):
    Position 0:  input=1    → target=567   (predict next)
    Position 1:  input=567  → target=892   (predict next)
    Position 2:  input=892  → target=123   (predict next)
    ...
    Position 6:  input=234  → target=1     (across boundary!)
    Position 7:  input=1    → target=345   (BOS helps here)

  CRITICAL: Token at position 6 (234) can attend to BOS at position 0
           (full document context), then predicts BOS at position 7
           (start of next document).

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 6: Memory Transfer to GPU

  📐 Code

  # Pre-allocated buffers (done ONCE)
  cpu_buffer = torch.empty(2 * B * T, dtype=torch.long, pin_memory=True)
  gpu_buffer = torch.empty(2 * B * T, dtype=torch.long, device="cuda")

  # Views into buffers
  cpu_inputs = cpu_buffer[:B * T].view(B, T)
  cpu_targets = cpu_buffer[B * T:].view(B, T)
  inputs = gpu_buffer[:B * T].view(B, T)
  targets = gpu_buffer[B * T:].view(B, T)

  # Each batch:
  cpu_inputs.copy_(row_buffer[:, :-1])   # CPU → CPU (fast)
  cpu_targets.copy_(row_buffer[:, 1:])   # CPU → CPU (fast)
  gpu_buffer.copy_(cpu_buffer, non_blocking=True)  # Async HtoD
  yield inputs, targets, state_dict

  🖼️ Visual: Memory Layout

  CPU MEMORY (Host)                              GPU MEMORY (Device)
  ┌─────────────────────────────────────┐        ┌─────────────────────────────────────┐
  │ cpu_buffer: [2*B*T] = [32] elements │        │ gpu_buffer: [32] elements on CUDA   │
  │  (page-locked / pinned memory)      │        │                                     │
  │                                     │        │                                     │
  │ ┌─────────────────────────────────┐ │        │ ┌─────────────────────────────────┐ │
  │ │ cpu_inputs:  [16] → view(2, 8)  │ │        │ │ inputs:  [16] → view(2, 8)      │ │
  │ │ [1, 567, 892, 123, 45, 678, ...]│ │        │ │ [1, 567, 892, 123, 45, 678, ...]│ │
  │ │ [1, 89, 4567, 2345, 1, 345, ...]│ │        │ │ [1, 89, 4567, 2345, 1, 345, ...]│ │
  │ └─────────────────────────────────┘ │        │ └─────────────────────────────────┘ │
  │                                     │        │                                     │
  │ ┌─────────────────────────────────┐ │        │ ┌─────────────────────────────────┐ │
  │ │ cpu_targets: [16] → view(2, 8)  │ │        │ │ targets: [16] → view(2, 8)      │ │
  │ │ [567, 892, 123, 45, 678, 234,..]│ │        │ │ [567, 892, 123, 45, 678, 234,..]│ │
  │ │ [89, 4567, 2345, 1, 345, 678,..]│ │        │ │ [89, 4567, 2345, 1, 345, 678,..]│ │
  │ └─────────────────────────────────┘ │        │ └─────────────────────────────────┘ │
  └─────────────────────────────────────┘        └─────────────────────────────────────┘
              │                                                  ▲
              │      gpu_buffer.copy_(cpu_buffer,                 │
              │                 non_blocking=True)               │
              └──────────────────────────────────────────────────┘

                     ASYNC DMA TRANSFER (NVLink/PCIe)
                     CPU continues immediately (prepares next batch)
                     while GPU receives data in background

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 7: Complete Data Flow Summary

  🖼️ End-to-End Visualization

  ┌─────────────────────────────────────────────────────────────────────────────┐
  │                        END-TO-END DATA FLOW                                  │
  └─────────────────────────────────────────────────────────────────────────────┘

  LEVEL 1: STORAGE (Parquet on Disk)
  ┌─────────────────────────────────────────────────────────────────────────────┐
  │ shard_00000.parquet (2GB)                                                   │
  │ ┌─────────────┬─────────────┬─────────────┬─────────────┐                  │
  │ │ Row Group 0 │ Row Group 1 │ Row Group 2 │     ...     │                  │
  │ │  ~10k docs  │  ~10k docs  │  ~10k docs  │             │                  │
  │ │  (GPU 0)    │   (GPU 1)   │   (GPU 2)   │             │                  │
  │ └──────┬──────┴──────┬──────┴──────┬──────┴─────────────┘                  │
  └────────┼─────────────┼─────────────┼────────────────────────────────────────┘
           │             │             │
           ▼             ▼             ▼

  LEVEL 2: DOCUMENT BATCHES (CPU RAM)
  ┌─────────────────────────────────────────────────────────────────────────────┐
  │ batch = ["The cat...", "In 1492...", "def hello():", ...]  # 128 strings   │
  │                                                                             │
  │ Tokenizer (4 threads):                                                      │
  │   "The cat..." → [1, 567, 892, 123, 45, 678, 234]          len=7           │
  │   "In 1492..." → [1, 89, 4567, 2345, 1234, 567]            len=6           │
  │   "def hello.."→ [1, 345, 678, 90]                         len=3           │
  │                                                                             │
  │ doc_buffer: [doc0(7), doc1(6), doc2(3), doc3(45), doc4(4), ...]  # 1000    │
  └─────────────────────────────────────────────────────────────────────────────┘
           │
           ▼

  LEVEL 3: PACKING ALGORITHM (CPU)
  ┌─────────────────────────────────────────────────────────────────────────────┐
  │                                                                             │
  │   ROW 0 (9 tokens):  [doc0(7) | doc2_crop(2)]                              │
  │                       [1,567,892,123,45,678,234, 1,345]                    │
  │                                                                             │
  │   ROW 1 (9 tokens):  [doc1(6) | doc4(4)_crop? no, 6+4=10>9                 │
  │                       [doc1(6) | doc5_crop...]                             │
  │                                                                             │
  │ row_buffer: [B=2, T+1=9]                                                    │
  │ ┌─────────────────────────────────────────────────────────┐                │
  │ │ [1, 567, 892, 123, 45, 678, 234, 1, 345]               │ ← Row 0        │
  │ │ [1, 89, 4567, 2345, 1, 345, 678, 90, 1]                │ ← Row 1        │
  │ └─────────────────────────────────────────────────────────┘                │
  └─────────────────────────────────────────────────────────────────────────────┘
           │
           ▼

  LEVEL 4: SLICING (CPU, contigous memory)
  ┌─────────────────────────────────────────────────────────────────────────────┐
  │                                                                             │
  │  row_buffer[:, :-1] → cpu_inputs    row_buffer[:, 1:] → cpu_targets         │
  │                                                                             │
  │  ┌────────────────────────┐        ┌────────────────────────┐              │
  │  │ inputs:                │        │ targets:               │              │
  │  │ [1, 567, 892, 123, ...│        │ [567, 892, 123, 45, ...│              │
  │  │ [1, 89, 4567, 2345,...│        │ [89, 4567, 2345, 1, ...│              │
  │  └────────────────────────┘        └────────────────────────┘              │
  │                                                                             │
  │  Copied into pinned cpu_buffer: [2, 8] each → flat [32]                    │
  └─────────────────────────────────────────────────────────────────────────────┘
           │
           ▼

  LEVEL 5: GPU TRANSFER (DMA)
  ┌─────────────────────────────────────────────────────────────────────────────┐
  │                                                                             │
  │  gpu_buffer.copy_(cpu_buffer, non_blocking=True)  ← Async transfer         │
  │                                                                             │
  │  GPU Memory (CUDA):                                                         │
  │  ┌─────────────────────────────────────────────────────────┐               │
  │  │ inputs:  torch.Size([2, 8])  on cuda:0                  │               │
  │  │ targets: torch.Size([2, 8])  on cuda:0                  │               │
  │  └─────────────────────────────────────────────────────────┘               │
  │                                                                             │
  │  Yield to training loop!                                                    │
  └─────────────────────────────────────────────────────────────────────────────┘

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  ⚠️ CRITICAL INVARIANTS CHECKLIST

   Check   Invariant                                          Why It Matters
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   ✅      Every row starts with BOS (token 1)                Model can always attend to sequence start
   ✅      inputs[i] == targets[i-1] for all i>0              Autoregressive property maintained
   ✅      No padding tokens (100% utilization)               Every GPU cycle trains on real data
   ✅      Documents are cropped, never truncated mid-token   Token boundaries preserved
   ✅      DDP shards by row_group, not document              Avoids loading imbalance
   ✅      Resume advances by 1 row_group                     Prevents data repetition on restart

  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  ❓ FINAL CHECK

  Before you build this for nanoseek, trace through mentally:

  Q: In the example above, token 345 appears twice:

  • Row 0, position 8: as target (predicted from 1)
  • Row 1, position 5: as input (context for 678)

  Is this token being trained on twice? Is that a problem?

  Think about it. I'll tell you if you're right when you reply.